# Local RAG for mathematical Markdown and Jupyter notebooks

This notebook indexes `.md` and `.ipynb` files beneath `math_docs` without changing the PDF-only `math_rag.py`. Notebook outputs and metadata are deliberately excluded; Markdown prose and, optionally, code cells are indexed with source/cell citations. Retrieval can be scoped with an include/exclude lexicon and is displayed compactly for inspection.

## 1. Configuration

Edit the paths or lexicon here. Leave both lexicon sets empty for ordinary semantic retrieval. Include terms narrow results to your subject vocabulary; exclude terms suppress unwanted senses or domains.

In [1]:
from __future__ import annotations

import hashlib
import html
import json
import re
from pathlib import Path

import chromadb
from IPython.display import Markdown, display
from langchain_text_splitters import RecursiveCharacterTextSplitter

SOURCE_DIR = Path("/Users/brad/Documents/Notes/math_docs")
DB_DIR = Path("/Users/brad/Documents/ChatGPT/JupyterNotebook Evals/chroma_math_notes")
COLLECTION_NAME = "math_notes_v1"
CHUNK_SIZE = 1200
CHUNK_OVERLAP = 180
INDEX_CODE_CELLS = True

# Examples only: {"topology", "homology"} or {"lorentz", "spacetime"}.
INCLUDE_TERMS: set[str] = set()
EXCLUDE_TERMS: set[str] = set()

assert SOURCE_DIR.is_dir(), f"Missing source directory: {SOURCE_DIR}"
print(f"Source: {SOURCE_DIR}\nDatabase: {DB_DIR}")

/Users/brad/Documents/DevOps_DSOps/.venv/lib/python3.14/site-packages/langchain_core/utils/pydantic.py:42: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1


Source: /Users/brad/Documents/Notes/math_docs
Database: /Users/brad/Documents/ChatGPT/JupyterNotebook Evals/chroma_math_notes


## 2. Extract clean, source-scoped text

Markdown is indexed as authored. For notebooks, only cell source is read—never saved output, execution counts, widgets, attachments, or notebook metadata. Each extracted unit retains its relative filename, cell number, and cell type.

In [2]:
def digest(path: Path) -> str:
    value = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            value.update(block)
    return value.hexdigest()


def discover_sources(root: Path) -> list[Path]:
    return sorted(
        path for path in root.rglob("*")
        if path.is_file()
        and path.suffix.lower() in {".md", ".ipynb"}
        and ".ipynb_checkpoints" not in path.parts
    )


def extract_units(path: Path, root: Path, include_code: bool = True) -> list[dict]:
    relative = str(path.relative_to(root))
    if path.suffix.lower() == ".md":
        text = path.read_text(encoding="utf-8", errors="replace").strip()
        return [{"text": text, "relative_source": relative,
                 "unit": 1, "unit_type": "markdown_file"}] if text else []

    notebook = json.loads(path.read_text(encoding="utf-8"))
    units = []
    for cell_number, cell in enumerate(notebook.get("cells", []), start=1):
        cell_type = cell.get("cell_type", "unknown")
        if cell_type not in {"markdown", "code"}:
            continue
        if cell_type == "code" and not include_code:
            continue
        source = cell.get("source", "")
        text = "".join(source) if isinstance(source, list) else str(source)
        text = text.strip()
        if text:
            units.append({"text": text, "relative_source": relative,
                          "unit": cell_number, "unit_type": cell_type})
    return units


sources = discover_sources(SOURCE_DIR)
display(Markdown(f"**Discovered:** {len(sources)} Markdown/notebook files"))
for path in sources[:20]:
    print(path.relative_to(SOURCE_DIR))

**Discovered:** 42 Markdown/notebook files

Bezout_Dixon_resultant.ipynb
ChASE-brunowu-gihub-README.md
Hermite-Polynomials_dindagustiayu_README.md
Integration.ipynb
IntegrationOverPolytopes.ipynb
Macaulay_resultant.ipynb
ODARI-CHARLESS1_Calculus-Notebooks_Readme.md
SymPlay.ipynb
bernstein-vazirani.ipynb
classiq-library-README.md
defining-quantum-circuits.ipynb
density.ipynb
deutsch-jozsa.ipynb
fidelity.ipynb
fresnel_integrals.ipynb
grover.ipynb
hidden-shift-problem.ipynb
identitysearch_example.ipynb
limit_examples_advanced.ipynb
plot_advanced.ipynb


## 3. Chunk and incrementally index

A separate Chroma collection prevents accidental mixing with PDF vectors. Stable hashes let reruns skip unchanged files. Deleted or renamed sources are cleaned from this collection. The default Chroma embedding model is downloaded once, then cached locally.

In [3]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n## ", "\n### " , "\n\n", "\n", ". ", " ", ""],
)



def open_collection(rebuild: bool = False):
    client = chromadb.PersistentClient(path=DB_DIR)
    if rebuild:
        try:
            client.delete_collection(COLLECTION_NAME)
        except Exception:
            pass
    store = client.get_or_create_collection(
        name=COLLECTION_NAME, metadata={"hnsw:space": "cosine"}
    )
    return client, store


def index_notes(rebuild: bool = False) -> dict:
    client, store = open_collection(rebuild=rebuild)
    paths = discover_sources(SOURCE_DIR)
    current = {str(path.relative_to(SOURCE_DIR)) for path in paths}
    stats = {"indexed": 0, "unchanged": 0, "chunks": 0, "removed": 0}

    for path in paths:
        relative = str(path.relative_to(SOURCE_DIR))
        file_hash = digest(path)
        old = store.get(where={"relative_source": relative}, include=["metadatas"])
        old_hash = old["metadatas"][0].get("file_sha256") if old["metadatas"] else None
        if old_hash == file_hash:
            stats["unchanged"] += 1
            continue
        if old["ids"]:
            store.delete(ids=old["ids"])

        rows = []
        for unit in extract_units(path, SOURCE_DIR, INDEX_CODE_CELLS):
            for chunk_number, text in enumerate(splitter.split_text(unit["text"])):
                chunk_id = f"{file_hash[:16]}:{unit['unit']}:{chunk_number}"
                metadata = {**{k: v for k, v in unit.items() if k != "text"},
                            "source": str(path), "file_sha256": file_hash,
                            "chunk": chunk_number}
                rows.append((chunk_id, text, metadata))
        for start in range(0, len(rows), 100):
            batch = rows[start:start + 100]
            store.upsert(ids=[r[0] for r in batch], documents=[r[1] for r in batch],
                         metadatas=[r[2] for r in batch])
        stats["indexed"] += 1
        stats["chunks"] += len(rows)

    known = store.get(include=["metadatas"])
    stale = [item_id for item_id, meta in zip(known["ids"], known["metadatas"])
             if meta["relative_source"] not in current]
    if stale:
        store.delete(ids=stale)
    stats["removed"] = len(stale)
    stats["total_chunks"] = store.count()
    return stats


# Set rebuild=True only after changing the embedding model or chunking rules.
index_stats = index_notes(rebuild=False)
display(index_stats)

{'indexed': 42,
 'unchanged': 0,
 'chunks': 1109,
 'removed': 0,
 'total_chunks': 1109}

## 4. Lexicon-scoped retrieval and safe display

Semantic search first retrieves a wider candidate set. Exact include/exclude terms then scope and lightly rerank those candidates. Matching is case-insensitive and respects word boundaries. The display escapes HTML and truncates long previews; this keeps source content from breaking the notebook layout.

In [7]:
def contains_term(text: str, term: str) -> bool:
    return re.search(rf"(?<!\w){re.escape(term)}(?!\w)", text, re.IGNORECASE) is not None


def retrieve(question: str, top_k: int = 6, candidate_k: int = 24,
             include_terms: set[str] | None = None,
             exclude_terms: set[str] | None = None) -> list[dict]:
    include = {t.strip() for t in (include_terms or set()) if t.strip()}
    exclude = {t.strip() for t in (exclude_terms or set()) if t.strip()}
    _, store = open_collection()
    if store.count() == 0:
        return []
    result = store.query(
        query_texts=[question], n_results=min(candidate_k, store.count()),
        include=["documents", "metadatas", "distances"],
    )
    hits = []
    for text, metadata, distance in zip(
        result["documents"][0], result["metadatas"][0], result["distances"][0]
    ):
        scope_text = f"{metadata['relative_source']}\n{text}"
        if exclude and any(contains_term(scope_text, term) for term in exclude):
            continue
        matches = sum(contains_term(scope_text, term) for term in include)
        if include and matches == 0:
            continue
        hits.append({"text": text, "metadata": metadata,
                     "distance": float(distance), "lexicon_matches": matches})
    hits.sort(key=lambda hit: (-hit["lexicon_matches"], hit["distance"]))
    return hits[:top_k]


def show_hits(hits: list[dict], preview_chars: int = 900) -> None:
    if not hits:
        display(Markdown("*No scoped matches. Index files or broaden the lexicon.*"))
        return
    for number, hit in enumerate(hits, start=1):
        meta = hit["metadata"]
        location = (f"cell {meta['unit']} ({meta['unit_type']})"
                    if meta['unit_type'] != "markdown_file" else "Markdown file")
        preview = hit["text"][:preview_chars]
        if len(hit["text"]) > preview_chars:
            preview += "…"
        safe_preview = html.escape(preview)
        display(Markdown(
            f"### [S{number}] `{meta['relative_source']}` — {location}\n"
            f"Distance: `{hit['distance']:.3f}` · Lexicon matches: "
            f"`{hit['lexicon_matches']}`\n\n<pre style='white-space:pre-wrap'>"
            f"{safe_preview}</pre>"
        ))


#UESTION = "Explain the main mathematical idea in these notes."
QUESTION = "Hermitian generator orthogonal properties."
hits = retrieve(QUESTION, include_terms=INCLUDE_TERMS, exclude_terms=EXCLUDE_TERMS)
show_hits(hits)

### [S1] `Hermite-Polynomials_dindagustiayu_README.md` — Markdown file
Distance: `0.557` · Lexicon matches: `0`

<pre style='white-space:pre-wrap'># Hermite Polynomials
Hermite polynomials, named after the French mathematician [Charles Hermite](https://en.wikipedia.org/wiki/Charles_Hermite), are orthogonal polynomials, in a sense to be described below, of the form

&lt;p align=&#x27;center&#x27;&gt;
    $$H_n(x) = (-1)^n e^{x^2} \frac{d^n}{dx^n} e^{-x^2} \quad (1)$$
&lt;/p&gt;

for $n=0, \ 1, \ 2, \ 3, \ldots$
The first few Hermite polynomials are
- for $n=0$ we have $H_0 (x) = 1$
- for $n=1$ we have $H_1 (x) = 2x$
- for $n=2$ we have $H_2 (x) = 4x^2 - 2$
- for $n=3$ we have $H_3 (x) = 8x^3 - 12x$
- for $n=4$ we have $H_4 (x) = 16x^4 - 48x^2 + 12$
- for $n=5$ we have $H_5 (x) = 32x^5 - 160x^3 + 120x$
- for $n=6$ we have $H_6 (x) = 64x^6 - 480x^4 + 720x^2 -120$
- for $n=7$ we have $H_7 (x) = 128x^7 - 1344x^5 + 3360x^3 - 1680x$

For $n \in \mathbb{N}$, we define Hermite polynomials $H_n (x)$ by
&lt;p align=&#x27;center&#x27;&gt;
    $$\sum_{n=0}^{\infty} \frac{H_n (x)}{n…</pre>

### [S2] `Hermite-Polynomials_dindagustiayu_README.md` — Markdown file
Distance: `0.578` · Lexicon matches: `0`

<pre style='white-space:pre-wrap'>## Quantum Mechanical Prerequisites
- Time-Independent $Schr\ddot{o}dinger$
- Normalization of wavefunctions ($|\psi (x)|^2$)
- Quantum number ($n \in {0, \ 1, \ 2, \ldots}$ ) and energy level ($E_n$).

# Quantum approach
In quantum mechanics and in other branches of physics, it is common to approach physical problems algebraics and analytic methods. Examples include the use of differential equations for many interesting models, the use of quantum groups in quantum physics, and of differential geometry in relativity theory. In this work, we discuss the Hermite polynomials, some of their properties and a brief description of their applications to the Quantum Harmonic Oscillator.

The Harmonic Oscillator&#x27;s Quantum Mechanical solution involves Hermite Polynomials, which are introduced here. The wavefunctions for the quantum harmonic oscillator contain the Gaussian form, which allows them to…</pre>

### [S3] `quantum-fourier-transform.ipynb` — cell 8 (markdown)
Distance: `0.587` · Lexicon matches: `0`

<pre style='white-space:pre-wrap'>$$
\begin{aligned}
QFT_N\vert x \rangle &amp; = \frac{1}{\sqrt{N}} \sum_{y=0}^{N-1}\omega_N^{xy} \vert y \rangle 
\\
&amp; = \frac{1}{\sqrt{N}} \sum_{y=0}^{N-1} e^{2 \pi i xy / 2^n} \vert y \rangle ~\text{since}\: \omega_N^{xy} = e^{2\pi i \frac{xy}{N}} \:\text{and}\: N = 2^n 
\\
&amp; = \frac{1}{\sqrt{N}} \sum_{y=0}^{N-1} e^{2 \pi i \left(\sum_{k=1}^n y_k/2^k\right) x} \vert y_1 \ldots y_n \rangle \:\text{rewriting in fractional binary notation}\: y = y_1\ldots y_n, y/2^n = \sum_{k=1}^n y_k/2^k 
\\
&amp; = \frac{1}{\sqrt{N}} \sum_{y=0}^{N-1} \prod_{k=1}^n e^{2 \pi i x y_k/2^k } \vert y_1 \ldots y_n \rangle \:\text{after expanding the exponential of a sum to a product of exponentials} 
\\
&amp; = \frac{1}{\sqrt{N}} \bigotimes_{k=1}^n  \left(\vert0\rangle + e^{2 \pi i x /2^k } \vert1\rangle \right) \:\text{after rearranging the sum and products, and expanding} 
\sum_{y=0}^{N-1} = \sum_{y_1=0}^{1}\sum_{y_2=0}…</pre>

### [S4] `quantum-walk-search-algorithm.ipynb` — cell 8 (markdown)
Distance: `0.590` · Lexicon matches: `0`

<pre style='white-space:pre-wrap'>Step 2(b) is equivalent to finding a unitary $R(P)$ that performs the following mapping: 
\begin{align}
\label{eq:mapping_1}
    \ket{U} &amp;\mapsto \ket{U}, \: \text{and} \\
    \ket{\psi} &amp;\mapsto -\ket{\psi}, \: \forall \ket{\psi} \text{in the span of eigenvectors of $W(P)$ that are orthogonal to $\ket{U}$}
\label{eq:mapping_2}
\end{align}</pre>

### [S5] `grover.ipynb` — cell 5 (markdown)
Distance: `0.619` · Lexicon matches: `0`

<pre style='white-space:pre-wrap'>$$
U_\omega|x\rangle = (-1)^{f(x)}|x\rangle
$$

and the oracle&#x27;s matrix will be a diagonal matrix of the form:

$$
U_\omega = 
\begin{bmatrix}
(-1)^{f(0)} &amp;   0         &amp; \cdots &amp;   0         \\
0           &amp; (-1)^{f(1)} &amp; \cdots &amp;   0         \\
\vdots      &amp;   0         &amp; \ddots &amp; \vdots      \\
0           &amp;   0         &amp; \cdots &amp; (-1)^{f(2^n-1)} \\
\end{bmatrix}
$$

&lt;details&gt;
    &lt;summary&gt;Circuit Construction of a Grover Oracle (click to expand)&lt;/summary&gt;
&lt;p&gt;
If we have our classical function $f(x)$, we can convert it to a reversible circuit of the form:
&lt;/p&gt;&lt;p&gt;
&lt;img alt=&quot;A Classical Reversible Oracle&quot; src=&quot;images/grover_boolean_oracle.svg&quot;&gt;
&lt;/p&gt;&lt;p&gt;
If we initialise the &#x27;output&#x27; qubit in the state $|{-}\rangle$, the phase kickback effect turns this into a Grover oracle (similar to the workings of the Deutsch-Jozsa oracle):
&lt;/p&gt;&lt;p&gt;  
&lt;img alt=&quot;Grover Oracle Constructed from a Classica…</pre>

### [S6] `quantum-walk-search-algorithm.ipynb` — cell 8 (markdown)
Distance: `0.619` · Lexicon matches: `0`

<pre style='white-space:pre-wrap'>\begin{equation}
    \ket{w} \ket{0} \mapsto \ket{w} \ket{\tilde{\theta_j}} \mapsto (-1)^{|\tilde{\theta_j} \neq 0|} \ket{w} \ket{\tilde{\theta_j}} \mapsto (-1)^{|\tilde{\theta_j} \neq 0|} \ket{w} \ket{0}
\end{equation}</pre>

## 5. Build a grounded prompt

This cell prepares bounded context for a local instruction model. It does not call a cloud service. Citations point to notebook cells or Markdown files rather than PDF pages.

In [8]:
def make_prompt(question: str, hits: list[dict]) -> str:
    context = "\n\n".join(
        f"[S{i}: {hit['metadata']['relative_source']}, "
        f"cell/unit {hit['metadata']['unit']}]\n{hit['text']}"
        for i, hit in enumerate(hits, start=1)
    )
    return f"""You are a careful mathematics tutor. Answer only from the supplied sources.
If the sources are insufficient, say so. Preserve mathematical notation, distinguish
assumptions from conclusions, and cite claims inline as [S1], [S2], etc.

SOURCES
{context}

QUESTION
{question}
"""

prompt = make_prompt(QUESTION, hits)
print(prompt[:5000])

You are a careful mathematics tutor. Answer only from the supplied sources.
If the sources are insufficient, say so. Preserve mathematical notation, distinguish
assumptions from conclusions, and cite claims inline as [S1], [S2], etc.

SOURCES
[S1: Hermite-Polynomials_dindagustiayu_README.md, cell/unit 1]
# Hermite Polynomials
Hermite polynomials, named after the French mathematician [Charles Hermite](https://en.wikipedia.org/wiki/Charles_Hermite), are orthogonal polynomials, in a sense to be described below, of the form

<p align='center'>
    $$H_n(x) = (-1)^n e^{x^2} \frac{d^n}{dx^n} e^{-x^2} \quad (1)$$
</p>

for $n=0, \ 1, \ 2, \ 3, \ldots$
The first few Hermite polynomials are
- for $n=0$ we have $H_0 (x) = 1$
- for $n=1$ we have $H_1 (x) = 2x$
- for $n=2$ we have $H_2 (x) = 4x^2 - 2$
- for $n=3$ we have $H_3 (x) = 8x^3 - 12x$
- for $n=4$ we have $H_4 (x) = 16x^4 - 48x^2 + 12$
- for $n=5$ we have $H_5 (x) = 32x^5 - 160x^3 + 120x$
- for $n=6$ we have $H_6 (x) = 64x^6 - 480x^4 + 720

## 6. Optional local generation

Uncomment and set `LOCAL_MODEL` to an absolute Hugging Face model directory. Keep retrieval inspection above as the primary quality check. A model identifier instead of a path can trigger a one-time network download.

In [6]:
# from transformers import AutoModelForCausalLM, AutoTokenizer
# LOCAL_MODEL = Path("/absolute/path/to/local/instruction-model")
# tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL)
# model = AutoModelForCausalLM.from_pretrained(LOCAL_MODEL, dtype="auto")
# model.to("mps")
# rendered = tokenizer.apply_chat_template(
#     [{"role": "user", "content": prompt}],
#     tokenize=False, add_generation_prompt=True,
# )
# inputs = tokenizer(rendered, return_tensors="pt").to(model.device)
# output = model.generate(**inputs, max_new_tokens=700, do_sample=False)
# answer_tokens = output[0, inputs["input_ids"].shape[1]:]
# display(Markdown(tokenizer.decode(answer_tokens, skip_special_tokens=True)))

## Operating notes

- Rerun the indexing cell after adding or editing source files; unchanged files are skipped.
- Set `INDEX_CODE_CELLS = False` and rebuild if prose-only retrieval is preferred.
- Rebuild after changing chunk size, code-cell policy, or embedding model.
- Lexicon filtering is intentionally transparent. It narrows candidate passages; it does not alter generated vocabulary.
- Keep `INCLUDE_TERMS` small. Over-scoping can hide relevant passages expressed with synonyms.
- Mathematical notebook outputs may contain useful results, but indexing them can duplicate prose, capture huge arrays, and preserve stale calculations. Summarize important results in a Markdown cell instead.